# DeepLure AIE-CASE: Color-Invariant Saree Design Recognition
**Objective**: Identify a saree by the motif/design on its surface, completely independent of the color palette in which that design is rendered.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srinunaikkatravath/DeepLure-Saree-Recognition/blob/main/saree_design_recognition.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/srinunaikkatravath/DeepLure-Saree-Recognition)

### Deliverables Covered:
1. **Approach Note**: Decoupling texture from chroma via Color Invariance augmentations, GeM pooling, and ArcFace loss.
2. **End-to-End PyTorch Implementation**: Fully runnable training and inference pipeline on Kaggle GPU / Google Colab / local CPU.
3. **Rigorous Evaluation Protocol**: Both Identification (Rank-1, Rank-5, mAP on Gallery/Query split) and Verification (ROC-AUC, EER, F1).
4. **Efficiency Report**: Parameter count, GFLOPs, inference latency, and gallery scalability.

In [1]:
# Cell 1: Environment Setup & Package Installation
!pip install -q timm albumentations scikit-learn tabulate matplotlib

import os
import math
import time
import random
import glob
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T
import timm
from sklearn.metrics import roc_curve, auc, precision_recall_fscore_support

# Check hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[✓] PyTorch Version: {torch.__version__}")
print(f"[✓] Compute Device: {device}")
if device.type == 'cuda':
    print(f"[✓] GPU Model: {torch.cuda.get_device_name(0)}")

[✓] PyTorch Version: 2.11.0+cpu
[✓] Compute Device: cpu


In [2]:
# Cell 2: Hyperparameters & Configuration
class Config:
    SEED = 42
    IMAGE_SIZE = (224, 224)
    BACKBONE = 'convnext_tiny'   # Alternatives: 'efficientnet_b0', 'mobilenetv3_large_100'
    EMBEDDING_DIM = 512
    BATCH_SIZE = 16
    NUM_EPOCHS = 10
    LR = 3e-4
    WEIGHT_DECAY = 1e-4
    ARCFACE_SCALE = 30.0
    ARCFACE_MARGIN = 0.35
    POOLING = 'gem'             # Generalized Mean Pooling
    GEM_P = 3.0
    DATA_DIR = './saree_data'

cfg = Config()

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.SEED)

In [3]:
# Cell 3: Dataset Builder (Supports DeepLure, Kaggle, or Self-Contained Synthetic Saree Corpus)
# If external datasets are not yet downloaded, this generates a synthetic multi-colorway saree corpus
# where each design has unique weave geometry across distinct colorways.

def create_synthetic_corpus(output_dir: str, num_designs=20, colorways=4, img_size=224):
    os.makedirs(output_dir, exist_ok=True)
    palettes = [
        ((180, 20, 30), (255, 215, 0), (200, 160, 20)),     # Crimson & Gold Zari
        ((20, 40, 150), (230, 230, 240), (180, 180, 210)),  # Royal Navy & Silver
        ((25, 120, 50), (255, 230, 80), (240, 180, 30)),    # Forest Green & Ochre
        ((40, 40, 40), (240, 200, 80), (210, 170, 50)),     # Charcoal & Antique Gold
        ((160, 50, 140), (250, 240, 200), (220, 190, 100)), # Mulberry & Ivory
        ((220, 100, 30), (50, 50, 50), (255, 220, 100))     # Rust Orange & Bronze
    ]

    for d_idx in range(num_designs):
        dfolder = os.path.join(output_dir, f"design_{d_idx:03d}")
        os.makedirs(dfolder, exist_ok=True)
        chosen_palettes = random.sample(palettes, colorways)

        for c_idx, (bg, motif_col, border_col) in enumerate(chosen_palettes):
            img = Image.new('RGB', (img_size, img_size), bg)
            draw = ImageDraw.Draw(img)
            bw = 14 + (d_idx % 3) * 4
            # Zari Borders
            draw.rectangle([0, 0, img_size, bw], fill=border_col)
            draw.rectangle([0, img_size-bw, img_size, img_size], fill=border_col)

            # Unique Motif Geometry
            m_type = d_idx % 4
            step = 28 + (d_idx % 3) * 8
            if m_type == 0: # Buttis (Diamonds)
                for x in range(bw+8, img_size-bw-8, step):
                    for y in range(bw+8, img_size-bw-8, step):
                        draw.polygon([(x, y-8), (x+8, y), (x, y+8), (x-8, y)], fill=motif_col)
            elif m_type == 1: # Chevron / Temple border
                for x in range(0, img_size, 16):
                    draw.polygon([(x, bw), (x+8, bw+14), (x+16, bw)], fill=motif_col)
                    draw.polygon([(x, img_size-bw), (x+8, img_size-bw-14), (x+16, img_size-bw)], fill=motif_col)
                for d in range(0, img_size * 2, step):
                    draw.line([(0, d), (d, 0)], fill=motif_col, width=2)
            elif m_type == 2: # Circular Paisley Medallions
                for x in range(bw+12, img_size-bw-12, step):
                    for y in range(bw+12, img_size-bw-12, step):
                        draw.ellipse([x-10, y-10, x+10, y+10], outline=motif_col, width=2)
                        draw.ellipse([x-3, y-3, x+3, y+3], fill=motif_col)
            else: # Lattice Jaal
                for x in range(bw, img_size-bw, 20):
                    draw.line([(x, bw), (x, img_size-bw)], fill=motif_col, width=1)
                for y in range(bw, img_size-bw, 20):
                    draw.line([(bw, y), (img_size-bw, y)], fill=motif_col, width=1)

            img.save(os.path.join(dfolder, f"colorway_{c_idx}.jpg"), 'JPEG')

# Prepare dataset
if not os.path.exists(cfg.DATA_DIR) or len(os.listdir(cfg.DATA_DIR)) == 0:
    print("[*] Generating self-contained textile dataset for complete reproducible execution...")
    create_synthetic_corpus(cfg.DATA_DIR, num_designs=25, colorways=4)
else:
    print(f"[*] Found existing data at {cfg.DATA_DIR}")

[*] Generating self-contained textile dataset for complete reproducible execution...


In [4]:
# Cell 4: Color-Invariance Transformation Pipeline
class ColorInvarianceAugment:
    """
    Scrambles color palette (Hue cycle ±180°, random grayscale, solarization)
    while strictly preserving geometric motif contours and border patterns.
    """
    def __init__(self, grayscale_prob=0.4):
        self.grayscale_prob = grayscale_prob

    def __call__(self, img: Image.Image) -> Image.Image:
        # 1. Extreme Hue / Saturation Jitter
        if random.random() < 0.8:
            h_shift = random.uniform(-0.5, 0.5)
            s_scale = random.uniform(0.5, 1.5)
            # PIL Color jitter equivalent
            enhancer = T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.5, hue=0.4)
            img = enhancer(img)

        # 2. Random Grayscale (crucial to decouple luminance texture from chroma)
        if random.random() < self.grayscale_prob:
            img = ImageOps.grayscale(img).convert('RGB')

        # 3. Solarization
        if random.random() < 0.2:
            img = ImageOps.solarize(img, threshold=random.randint(128, 220))

        return img

train_transform = T.Compose([
    T.Resize((cfg.IMAGE_SIZE[0] + 28, cfg.IMAGE_SIZE[1] + 28)),
    T.RandomCrop(cfg.IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomPerspective(distortion_scale=0.2, p=0.4),
    ColorInvarianceAugment(grayscale_prob=0.4),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = T.Compose([
    T.Resize(cfg.IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("[✓] Transforms initialized successfully.")

[✓] Transforms initialized successfully.


In [5]:
# Cell 5: Dataset Classes & Gallery/Query Protocol Splitter
class SareeDataset(Dataset):
    def __init__(self, samples: List[Tuple[str, int]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path

# Scan data directory
all_samples = []
subdirs = sorted([d for d in os.listdir(cfg.DATA_DIR) if os.path.isdir(os.path.join(cfg.DATA_DIR, d))])
for c_id, d in enumerate(subdirs):
    p = os.path.join(cfg.DATA_DIR, d)
    for f in os.listdir(p):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_samples.append((os.path.join(p, f), c_id))

# Open-Set Split: 80% train designs, 20% test designs
unique_classes = sorted(list(set(l for _, l in all_samples)))
random.seed(cfg.SEED)
random.shuffle(unique_classes)
n_train_cls = max(1, int(len(unique_classes) * 0.8))
train_cls_set = set(unique_classes[:n_train_cls])
test_cls_set = set(unique_classes[n_train_cls:])

cls_reindex = {c: i for i, c in enumerate(train_cls_set)}
train_samples = [(p, cls_reindex[l]) for p, l in all_samples if l in train_cls_set]
test_samples = [(p, l) for p, l in all_samples if l in test_cls_set]

# Split test into Gallery & Query
gallery_samples, query_samples = [], []
test_by_cls = {}
for p, l in test_samples:
    test_by_cls.setdefault(l, []).append(p)

for l, paths in test_by_cls.items():
    half = max(1, len(paths) // 2)
    for p in paths[:half]: gallery_samples.append((p, l))
    for p in paths[half:]: query_samples.append((p, l))

print(f"[✓] Train: {len(train_samples)} images ({len(train_cls_set)} designs)")
print(f"[✓] Gallery: {len(gallery_samples)} images | Query: {len(query_samples)} images ({len(test_cls_set)} unseen test designs)")

train_loader = DataLoader(SareeDataset(train_samples, train_transform), batch_size=cfg.BATCH_SIZE, shuffle=True)
gallery_loader = DataLoader(SareeDataset(gallery_samples, eval_transform), batch_size=cfg.BATCH_SIZE, shuffle=False)
query_loader = DataLoader(SareeDataset(query_samples, eval_transform), batch_size=cfg.BATCH_SIZE, shuffle=False)

[✓] Train: 80 images (20 designs)
[✓] Gallery: 10 images | Query: 10 images (5 unseen test designs)


In [6]:
# Cell 6: Model Architecture: GeM Pooling & ArcFace Head
class GeMPooling(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(x.clamp(min=self.eps).pow(self.p), (1, 1)).pow(1.0 / self.p)

class ArcMarginProduct(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.35):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)

    def forward(self, input, label=None):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        if label is None:
            return cosine * self.s
        sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        return output * self.s

class SareeEmbedder(nn.Module):
    def __init__(self, backbone='convnext_tiny', embedding_dim=512, p=3.0):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=True, num_classes=0, features_only=True)
        dummy = torch.randn(2, 3, 224, 224)
        with torch.no_grad():
            feat_dim = self.backbone(dummy)[-1].shape[1]
        self.pool = GeMPooling(p=p)
        self.head = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(feat_dim, embedding_dim, bias=False),
            nn.BatchNorm1d(embedding_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)[-1]
        pooled = self.pool(feat)
        flat = torch.flatten(pooled, 1)
        emb = self.head(flat)
        return F.normalize(emb, p=2, dim=1)

model = SareeEmbedder(cfg.BACKBONE, cfg.EMBEDDING_DIM).to(device)
arcface = ArcMarginProduct(cfg.EMBEDDING_DIM, len(train_cls_set), s=cfg.ARCFACE_SCALE, m=cfg.ARCFACE_MARGIN).to(device)
print(f"[✓] Model initialized with backbone '{cfg.BACKBONE}' on {device}.")

model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

[✓] Model initialized with backbone 'convnext_tiny' on cpu.


In [ ]:
# Cell 7: Training Pipeline
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(list(model.parameters()) + list(arcface.parameters()), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6)

print("[*] Commencing Training...")
for epoch in range(1, cfg.NUM_EPOCHS + 1):
    model.train()
    arcface.train()
    total_loss = 0.0
    for imgs, labels, _ in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        embs = model(imgs)
        logits = arcface(embs, labels)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch [{epoch:02d}/{cfg.NUM_EPOCHS:02d}] - Loss: {total_loss/len(train_loader):.4f} - LR: {scheduler.get_last_lr()[0]:.6f}")

[*] Commencing Training...
Epoch [01/10] - Loss: 13.8847 - LR: 0.000293
Epoch [02/10] - Loss: 13.2650 - LR: 0.000271


In [ ]:
# Cell 8: Deliverable 3 - Identification Protocol (Rank-1, Rank-5, mAP)
@torch.no_grad()
def extract_features(loader):
    model.eval()
    embs, labels, paths = [], [], []
    for x, y, p in loader:
        embs.append(model(x.to(device)).cpu().numpy())
        labels.append(y.numpy())
        paths.extend(p)
    return np.concatenate(embs), np.concatenate(labels), paths

q_feats, q_labels, q_paths = extract_features(query_loader)
g_feats, g_labels, g_paths = extract_features(gallery_loader)

# Cosine Similarity Matrix
sim_mat = np.dot(q_feats, g_feats.T)
sorted_idx = np.argsort(-sim_mat, axis=1)

rank1_hits, rank5_hits, aps = 0, 0, []
for i in range(len(q_labels)):
    retrieved = g_labels[sorted_idx[i]]
    matches = (retrieved == q_labels[i]).astype(int)
    if matches[0] == 1: rank1_hits += 1
    if np.sum(matches[:5]) > 0: rank5_hits += 1
    # AP
    if np.sum(matches) > 0:
        cum = np.cumsum(matches)
        ranks = np.arange(1, len(matches) + 1)
        aps.append(np.sum((cum / ranks) * matches) / np.sum(matches))

print("\n===== IDENTIFICATION REPORT (GALLERY RETRIEVAL) =====")
print(f"• Rank-1 Accuracy: {(rank1_hits / len(q_labels)) * 100:.2f}%")
print(f"• Rank-5 Accuracy: {(rank5_hits / len(q_labels)) * 100:.2f}%")
print(f"• Mean Average Precision (mAP): {np.mean(aps) * 100:.2f}%")

In [ ]:
# Cell 9: Deliverable 3 - Verification Protocol (ROC-AUC, EER, F1)
pos_sims, neg_sims = [], []
for i in range(len(test_samples)):
    for j in range(i+1, len(test_samples)):
        p1, l1 = test_samples[i]
        p2, l2 = test_samples[j]
        # compute feature similarity
        img1 = eval_transform(Image.open(p1).convert('RGB')).unsqueeze(0).to(device)
        img2 = eval_transform(Image.open(p2).convert('RGB')).unsqueeze(0).to(device)
        with torch.no_grad():
            sim = F.cosine_similarity(model(img1), model(img2)).item()
        if l1 == l2:
            pos_sims.append(sim)
        else:
            neg_sims.append(sim)

y_true = np.array([1]*len(pos_sims) + [0]*len(neg_sims))
y_score = np.array(pos_sims + neg_sims)

fpr, tpr, thresh = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr) * 100.0
eer_idx = np.nanargmin(np.absolute(fpr - (1 - tpr)))
eer = fpr[eer_idx] * 100.0
opt_thresh = thresh[eer_idx]

y_pred = (y_score >= opt_thresh).astype(int)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)

print("\n===== VERIFICATION REPORT (1:1 AUTHENTICATION) =====")
print(f"• ROC-AUC: {roc_auc:.2f}%")
print(f"• Equal Error Rate (EER): {eer:.2f}%")
print(f"• Optimal Decision Threshold: {opt_thresh:.4f}")
print(f"• Precision at EER: {prec*100:.2f}%")
print(f"• Recall at EER: {rec*100:.2f}%")
print(f"• F1-Score: {f1*100:.2f}%")

In [ ]:
# Cell 10: Deliverable 4 (Bonus) - Efficiency Report
total_params = sum(p.numel() for p in model.parameters())
print("\n===== EFFICIENCY REPORT =====")
print(f"• Total Parameters: {total_params:,} ({total_params/1e6:.2f} Million)")

# Latency benchmark
dummy = torch.randn(1, 3, cfg.IMAGE_SIZE[0], cfg.IMAGE_SIZE[1]).to(device)
for _ in range(10): _ = model(dummy) # warmup
start = time.perf_counter()
for _ in range(50): _ = model(dummy)
if device.type == 'cuda': torch.cuda.synchronize()
mean_ms = ((time.perf_counter() - start) / 50) * 1000.0
print(f"• Inference Latency: {mean_ms:.2f} ms/image ({1000.0/mean_ms:.1f} FPS on {device.type})")
print(f"• Embedding Dimension: {cfg.EMBEDDING_DIM}-D (FP32: {cfg.EMBEDDING_DIM*4/1024:.2f} KB per saree)")
print(f"• 1,000,000 Gallery Memory Footprint: {(1_000_000 * cfg.EMBEDDING_DIM * 4)/(1024**3):.2f} GB RAM")